# Creating an Arbitrary Index from Nasdaq Company Tickers

The goal of this project is to create an arbitrary stock index based on Nasdaq-listed companies whose ticker symbols follow a specific rule.

For example, the **Y Index** contains all eligible Nasdaq-listed companies whose ticker starts with the letter **Y**.

The objective is not only to create the index, but also to build a reproducible pipeline that:

1. Downloads the universe of Nasdaq-listed securities.
2. Identifies real operating companies.
3. Removes financial instruments that are not suitable for an equity index.
4. Applies eligibility rules similar to professional index providers.
5. Calculates index constituents and weights.
6. Performs exploratory data analysis (EDA) on the resulting index.

---

## 1. Data Sources

### SEC Company Tickers

The SEC provides a list of companies that file reports with the U.S. Securities and Exchange Commission.

Source:

https://www.sec.gov/files/company_tickers.json

This dataset contains:

- Company name
- Ticker symbol
- SEC CIK identifier

Example:

```json
{
"ticker": "AAPL",
"title": "Apple Inc.",
"cik_str": 320193
}
```

The SEC dataset is useful because it represents **actual reporting companies**, unlike exchange symbol lists that may contain many financial products.

---

### Nasdaq Listed Securities

Source:

https://ftp.nasdaqtrader.com/SymbolDirectory/nasdaqlisted.txt

This file contains all securities listed on Nasdaq.

It includes:

- Common stocks
- ADRs
- ETFs
- Warrants
- Rights
- Units
- Preferred shares
- Other financial instruments

The Nasdaq dataset is required because it provides the official universe of securities traded on Nasdaq.

### Data location

Both files are in the `/data` directory.

---

## 2. Building the Company Universe

We combine the SEC and Nasdaq datasets to create a clean universe of companies.

The process is:


Nasdaq Listed Securities --> Remove ETFs and test securities --> Join with SEC reporting companies -- >
Remove non-equity securities --> Remove SPACs --> Apply company age filters (remove less than 1 year)

This gives us the universe where we can select our companies for the index.

---

## 3. Security Filtering Rules

We exclude the following securities because they do not represent ordinary company ownership.

- ETFs
- Test issues
- NextShares
- Warrants
- Rights
- Units
- Preferred shares
- Bonds and notes
- Other derivative instruments

Examples:

| Symbol | Type | Action |
|---|---|---|
| AAPL | Common stock | Keep |
| YICC | Common stock | Keep |
| YICCU | Unit | Remove |
| YICCW | Warrant | Remove |

---

## 4. Company Eligibility Rules

We apply other filters to create a more realistic index universe:

### Remove SPACs

We remove Special Purpose Acquisition Companies.

Examples of excluded names:

- Acquisition Corp
- Acquisition Holdings
- Blank Check Company

The reason is that these entities do not represent mature operating businesses.

---

### Minimum Company Age

We exclude companies younger than one yeaare excluded.

This avoids including:

- Recently created SPACs
- Very recent IPOs
- **Companies without sufficient trading history**

---

## 5. Creating the Letter-Based Index Universe

After cleaning the Nasdaq company universe, we group companies by ticker letter.

- A Index
- J Index
- Y Index
- Any other ticker-based universe

## GOAL: Find the Most Underrepresented Letter in the tickers, and Create an Index Based on it

In [9]:
import json
import pandas as pd
from datetime import datetime, timedelta
from openbb import obb


In [ ]:
# -------------------------------------------------------
# Loading
# -------------------------------------------------------


def load_sec_companies(path):
    """Load SEC company_tickers.json."""

    with open(path, "r") as f:
        sec = json.load(f)

    df = pd.DataFrame.from_dict(sec, orient="index").rename(
        columns={
            "ticker": "Symbol",
            "title": "Company",
            "cik_str": "CIK",
        }
    )

    df["Symbol"] = df["Symbol"].str.upper()

    return df


def load_nasdaq(path):
    """Load nasdaqlisted.txt."""

    df = pd.read_csv(path, sep="|")

    df = df[df["Symbol"] != "File Creation Time"]

    df = df[(df["ETF"] == "N") & (df["NextShares"] == "N") & (df["Test Issue"] == "N")]

    return df


# -------------------------------------------------------
# Cleaning
# -------------------------------------------------------


def remove_spacs(df):
    """
    Remove SPACs / blank check companies.
    """

    patterns = [
        " acquisition ",
        " acquisition$",
        " acquisition corp",
        " acquisition corporation",
        " acquisition holdings",
        " blank check",
    ]

    regex = "|".join(patterns)

    mask = ~df["Company"].str.lower().str.contains(
        regex,
        regex=True,
        na=False,
    )

    return df[mask]


def merge_company_universe(sec_df, nasdaq_df):
    """
    Keep only SEC-reporting companies that trade on Nasdaq.
    """

    df = nasdaq_df.merge(
        sec_df,
        on="Symbol",
        how="inner",
    )

    return df


# -------------------------------------------------------
# Cleaning securities
# -------------------------------------------------------


def remove_non_equity_securities(df):
    """
    Remove securities that are not common equity.
    """

    bad_patterns = [
        "warrant",
        "rights?",
        "unit",
        "preferred",
        "depositary",
        "note",
        "bond",
        "debenture",
    ]

    regex = "|".join(bad_patterns)

    mask = ~df["Security Name"].str.lower().str.contains(
        regex,
        regex=True,
        na=False,
    )

    return df[mask]


# -------------------------------------------------------
# Queries
# -------------------------------------------------------


def companies_by_letter(df, letter):
    return (
        df[df["Symbol"].str.startswith(letter.upper())]
        .sort_values("Symbol")
        .reset_index(drop=True)
    )


def ticker_letter_counts(df):
    """
    Count companies by first ticker letter.
    """

    counts = (
        df.assign(first=df["Symbol"].str[0])
        .groupby("first")
        .size()
        .sort_values()
        .rename("companies")
        .reset_index()
        .rename(columns={"first": "letter"})
    )

    return counts


def least_common_letters(df, n=10):
    return ticker_letter_counts(df).head(n)


In [ ]:
from datetime import timedelta


def has_one_year_history(ticker, minimum_days=365):
    """
    Return True if the ticker has at least one natural year of
    historical price data.

    Downloads two years of history to avoid Yahoo's default
    one-year window.
    """

    try:
        prices = obb.equity.price.historical(
            ticker,
            provider="yfinance",
            start_date=(datetime.today() - timedelta(days=730)).strftime("%Y-%m-%d"),
        ).to_df()

        if prices.empty:
            return False

        first_date = pd.to_datetime(prices.index.min())
        last_date = pd.to_datetime(prices.index.max())

        history_length = (last_date - first_date).days

        return history_length >= minimum_days

    except Exception:
        return False


def remove_young_companies(df, minimum_days=365):
    """
    Keep only companies with at least one natural year of
    historical price data.
    """

    valid = []

    for ticker in df["Symbol"]:
        has_history = has_one_year_history(
            ticker,
            minimum_days=minimum_days,
        )

        print(f"{ticker}: {'OK' if has_history else 'Too recent'}")

        if has_history:
            valid.append(ticker)

    return df[df["Symbol"].isin(valid)].reset_index(drop=True)

In [12]:
# -------------------------------------------------------
# Main
# -------------------------------------------------------


def build_company_universe(sec_json_path, nasdaq_path):
    """
    Build the clean Nasdaq company universe.
    """

    sec = load_sec_companies(sec_json_path)

    nasdaq = load_nasdaq(nasdaq_path)

    universe = merge_company_universe(sec, nasdaq)

    universe = remove_spacs(universe)

    universe = remove_non_equity_securities(universe)

    return universe

In [13]:
universe = build_company_universe(
    "/home/eric/Documents/mess_box/y_index/data/sec.json",
    "/home/eric/Documents/mess_box/y_index/data/nasdaqlisted.txt",
)

# Count by initial letter
counts = ticker_letter_counts(universe)

print(counts)

# Least common letters
least_common_letters(universe)

# Most common letters
counts.sort_values("companies", ascending=False)

   letter  companies
0       Y         17
1       Q         27
2       Z         28
3       X         30
4       J         33
5       U         37
6       K         63
7       W         66
8       V         76
9       O         88
10      D         93
11      H        101
12      E        112
13      L        123
14      G        123
15      I        125
16      R        126
17      T        137
18      N        137
19      F        137
20      B        157
21      M        163
22      P        177
23      S        245
24      C        289
25      A        307


,letter,companies
25,A,307
24,C,289
23,S,245
22,P,177
21,M,163
20,B,157
18,N,137
17,T,137
19,F,137
16,R,126


In [14]:
def companies_starting_with(df, letter):
    return (
        df[df["Symbol"].str.startswith(letter.upper())][["Symbol", "Company"]]
        .sort_values("Symbol")
        .reset_index(drop=True)
    )

In [15]:
y_index = remove_young_companies(companies_starting_with(universe, "Y"))

YAAS 2025-07-25 00:00:00
YDDL 2025-10-09 00:00:00
YDES 2025-08-26 00:00:00
YDKG 2025-07-25 00:00:00
YHC 2025-07-25 00:00:00
YHGJ 2025-07-25 00:00:00
YIBO 2025-07-25 00:00:00
YICC 2026-07-13 00:00:00
YJ 2025-07-25 00:00:00
YMAT 2025-07-30 00:00:00
YORW 2025-07-25 00:00:00
YSWY 2026-04-22 00:00:00
YSXT 2025-07-25 00:00:00
YTRA 2025-07-25 00:00:00
YXT 2025-07-25 00:00:00
YYAI 2025-07-25 00:00:00
YYGH 2025-07-25 00:00:00


## 

In [16]:
y_index

,Symbol,Company
0,YAAS,Youxin Technology Ltd
3,YDKG,Yueda Digital Holding
4,YHC,LQR House Inc.
5,YHGJ,YUNHONG GREEN CTI LTD.
6,YIBO,Planet Image International Ltd
8,YJ,Yunji Inc.
10,YORW,YORK WATER CO
12,YSXT,"YSX Tech Co., Ltd"
13,YTRA,"Yatra Online, Inc."
14,YXT,YXT.COM GROUP HOLDING Ltd
